# Course Generation Algorithm — Hackable Notebook

Reimplements ProfDeck's lecture-generation pipeline (**handbook course → lecture plan → per-lecture outline/slides/rubric evaluation**)
as a plain, editable script for prompt-engineering experiments.

- **Step 0** loads the MSc handbook PDF and lets you pick a course section (default: **4.1 Software Engineering Modules**, from page 23).
- **Step 0.5** extracts or generates **10–12 lectures**, saves JSON + Markdown.
- **Steps 1–3** define outline / slides / rubric prompts and generators.
- **Step 3.5** improves problem slides and re-evaluates (target **95**, max **3** rounds). Saves **publication-quality figures** (PDF/PNG/SVG).
- **Step 4** loops over every lecture: outline → slides → rubric eval → improve → CQI; saves JSON + PDF report.
- One `call_llm()` function talks to **OpenRouter** or **Anthropic** (from `.env`).

Install deps once: `pip install -r notebook-requirements.txt`

Run cells top to bottom.

In [1]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import requests
import seaborn as sns
from fpdf import FPDF
from IPython.display import HTML, display
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.util import Inches, Pt
from pypdf import PdfReader

NOTEBOOK_DIR = Path.cwd()


def _load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


_load_dotenv(NOTEBOOK_DIR / ".env")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
LLM_PROVIDER = "openrouter" if OPENROUTER_API_KEY else "anthropic" if ANTHROPIC_API_KEY else ""
if not LLM_PROVIDER:
    raise EnvironmentError(
        "Set OPENROUTER_API_KEY or ANTHROPIC_API_KEY in the environment or .env file."
    )

# OpenRouter slug; Anthropic fallback maps this to ANTHROPIC_TEXT_MODEL.
MODEL = "anthropic/claude-sonnet-4.5"
print(f"LLM provider: {LLM_PROVIDER} | model: {MODEL}")

LLM provider: openrouter | model: anthropic/claude-sonnet-4.5


## Step 0 — Course selection from CSSE Handbook

Reads `CSSE_MSc_Handbook_ Fall 2025_V1_3.pdf` from the project root and extracts the selected course section.

**Change your course:** edit `SELECTED_COURSE_ID` (and optionally `SELECTED_MODULE_CODE`) in the next cell, then re-run it.

| ID | Section | Pages |
|----|---------|-------|
| `4.1` | Software Engineering Modules | 23–35 |
| `4.2` | Cybersecurity Modules | 36–49 |
| `4.3` | Artificial Intelligence Modules | 50–59 |

More sections can be added to `HANDBOOK_SECTIONS` as you expand experiments.

In [2]:
# --- Handbook & course selection (edit these) ---
HANDBOOK_FILENAME = "CSSE_MSc_Handbook_ Fall 2025_V1_3.pdf"
SELECTED_COURSE_ID = "4.1"  # e.g. "4.2" Cybersecurity, "4.3" AI
SELECTED_MODULE_CODE = None  # e.g. "2025-MCSSE-SE-01"; None = first module in section

HANDBOOK_SECTIONS = {
    "4.1": {
        "title": "Software Engineering Modules",
        "start_page": 23,
        "end_page": 35,
    },
    "4.2": {
        "title": "Cybersecurity Modules",
        "start_page": 36,
        "end_page": 49,
    },
    "4.3": {
        "title": "Artificial Intelligence Modules",
        "start_page": 50,
        "end_page": 59,
    },
}




def extract_handbook_pages(pdf_path: Path, start_page: int, end_page: int) -> str:
    """Extract text from inclusive 1-based PDF page range."""
    reader = PdfReader(str(pdf_path))
    chunks: list[str] = []
    for page_num in range(start_page, min(end_page, len(reader.pages)) + 1):
        text = reader.pages[page_num - 1].extract_text() or ""
        chunks.append(f"--- Page {page_num} ---\n{text.strip()}")
    return "\n\n".join(chunks)


def parse_handbook_modules(section_text: str) -> list[dict]:
    """Parse module blocks from a handbook section (Module Name / Module Code / ECTS)."""
    modules: list[dict] = []
    for match in re.finditer(
        r"Module Name\s+(.+?)\s+Module Code\s+(\S+)\s+Module ECTS\s+(\d+)",
        section_text,
        flags=re.DOTALL,
    ):
        name = re.sub(r"\s+", " ", match.group(1)).strip()
        modules.append(
            {
                "name": name,
                "code": match.group(2),
                "ects": int(match.group(3)),
            }
        )
    return modules


handbook_path = NOTEBOOK_DIR / HANDBOOK_FILENAME
if not handbook_path.exists():
    raise FileNotFoundError(
        f"Handbook not found: {handbook_path}\n"
        "Place CSSE_MSc_Handbook_ Fall 2025_V1_3.pdf in the project root."
    )

if SELECTED_COURSE_ID not in HANDBOOK_SECTIONS:
    raise ValueError(
        f"Unknown SELECTED_COURSE_ID={SELECTED_COURSE_ID!r}. "
        f"Choose one of: {', '.join(HANDBOOK_SECTIONS)}"
    )

selected_section = HANDBOOK_SECTIONS[SELECTED_COURSE_ID]
handbook_section_text = extract_handbook_pages(
    handbook_path,
    selected_section["start_page"],
    selected_section["end_page"],
    )
handbook_modules = parse_handbook_modules(handbook_section_text)

if not handbook_modules:
    raise RuntimeError(
        f"No modules parsed in section {SELECTED_COURSE_ID}. "
        "Check page range or PDF text extraction."
    )

if SELECTED_MODULE_CODE:
    selected_module = next(
        (m for m in handbook_modules if m["code"] == SELECTED_MODULE_CODE),
        None,
    )
    if not selected_module:
        raise ValueError(
            f"Module {SELECTED_MODULE_CODE!r} not found. "
            f"Available: {[m['code'] for m in handbook_modules]}"
        )
else:
    selected_module = handbook_modules[0]

COURSE_TITLE = f"{SELECTED_COURSE_ID} {selected_section['title']}"
COURSE_CONTEXT = (
    f"MSc CSSE handbook section: {COURSE_TITLE}\n"
    f"Selected module: {selected_module['name']} ({selected_module['code']}, "
    f"{selected_module['ects']} ECTS)\n\n"
    f"{handbook_section_text[:12000]}"
    )

display(HTML(f"<h3>Handbook: {HANDBOOK_FILENAME}</h3>"))
display(
    HTML(
        "<dl>"
        f"<dt><b>Course section</b></dt><dd>{COURSE_TITLE}</dd>"
        f"<dt><b>Pages</b></dt><dd>{selected_section['start_page']}–{selected_section['end_page']}</dd>"
        f"<dt><b>Selected module</b></dt><dd>{selected_module['name']} ({selected_module['code']})</dd>"
        "</dl>"
    )
    )
rows = "".join(
    f"<tr><td>{m['code']}</td><td>{m['name']}</td><td>{m['ects']}</td></tr>"
    for m in handbook_modules
    )
display(
    HTML(
        "<table border=1 cellpadding=6 style='border-collapse:collapse'>"
        "<tr><th>Module code</th><th>Module name</th><th>ECTS</th></tr>"
        f"{rows}</table>"
    )
    )
print(f"Using module: {selected_module['code']} — {selected_module['name']}")

Module code,Module name,ECTS
2025-MCSSE-SE-01,"Software Construction, Architecture and Engineering",5
2025-MCSSE-SE-02,Quality Engineering,5
2025-MCSSE-SE-03,Architectural Strategy,5
2025-MCSSE-SE-04,Advances in Software Engineering,5
2025-MDE-CS-04,Advanced Databases,5


Using module: 2025-MCSSE-SE-01 — Software Construction, Architecture and Engineering


## The one LLM call

Every pipeline step funnels through `call_llm()`. Change the provider/model here and it propagates everywhere.

In [3]:
JSON_ONLY_SUFFIX = (
    "\n\nYou MUST respond with valid JSON only — no markdown, "
    "no code fences, no commentary before or after."
)


def _anthropic_model_name(model: str) -> str:
    if os.environ.get("ANTHROPIC_TEXT_MODEL"):
        return os.environ["ANTHROPIC_TEXT_MODEL"]
    slug = model.split("/")[-1] if "/" in model else model
    if "claude-sonnet-4" in slug:
        return "claude-sonnet-4-5-20250929"
    return slug


def call_llm(
    system_prompt: str,
    user_prompt: str,
    *,
    temperature: float = 0.5,
    max_tokens: int = 8192,
    model: str = MODEL,
) -> str:
    """Single entry point for every LLM call in this notebook."""
    print(f"    -> calling {model} via {LLM_PROVIDER} (temperature={temperature}, max_tokens={max_tokens})...")
    start = time.time()
    system = system_prompt + JSON_ONLY_SUFFIX

    if LLM_PROVIDER == "openrouter":
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user_prompt},
                ],
            },
            timeout=180,
        )
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
    else:
        anthropic_model = _anthropic_model_name(model)
        response = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "x-api-key": ANTHROPIC_API_KEY,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": anthropic_model,
                "max_tokens": max_tokens,
                "temperature": temperature,
                "system": system,
                "messages": [{"role": "user", "content": user_prompt}],
            },
            timeout=180,
        )
        response.raise_for_status()
        payload = response.json()
        content = "".join(
            block.get("text", "")
            for block in payload.get("content", [])
            if block.get("type") == "text"
        )

    elapsed = time.time() - start
    print(f"    <- received response in {elapsed:.1f}s ({len(content)} chars)")
    return _strip_json_fences(content)


def _strip_json_fences(text: str) -> str:
    text = text.strip()
    match = re.match(r"^```(?:json)?\s*([\s\S]*?)```$", text)
    if match:
        return match.group(1).strip()
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if match:
        return match.group(1).strip()
    return text


def parse_with_retry(generate_fn, max_retries: int = 3, label: str = "") -> dict:
    """Calls generate_fn() (returns a JSON string), retrying by re-calling the LLM on parse failure."""
    prefix = f"[{label}] " if label else ""
    last_error = None
    raw = None
    for attempt in range(max_retries):
        print(f"{prefix}attempt {attempt + 1}/{max_retries}")
        raw = generate_fn()
        try:
            result = json.loads(raw)
            print(f"{prefix}parsed valid JSON on attempt {attempt + 1}")
            return result
        except json.JSONDecodeError as e:
            last_error = e
            print(f"{prefix}attempt {attempt + 1} failed to parse JSON: {e}")
    raise RuntimeError(
        f"Failed to parse JSON after {max_retries} attempts: {last_error}\nLast raw output:\n{raw}"
    )


## Score progress figures (publication quality)

Uses **matplotlib + seaborn** for journal-ready figures (vector PDF/SVG + 300 dpi PNG).

Install once: `pip install -r notebook-requirements.txt`

Other notebook output still uses plain HTML lists and tables.

In [4]:
def show_title(text: str, level: int = 4) -> None:
    display(HTML(f"<h{level}>{text}</h{level}>"))


def show_list(items: list, title: str | None = None) -> None:
    html = f"<b>{title}</b>" if title else ""
    html += "<ul>" + "".join(f"<li>{item}</li>" for item in items) + "</ul>"
    display(HTML(html))


def show_table(rows: list[list], headers: list[str]) -> None:
    html = "<table border=1 cellpadding=6 style=\'border-collapse:collapse\'><tr>"
    html += "".join(f"<th>{h}</th>" for h in headers) + "</tr>"
    for row in rows:
        html += "<tr>" + "".join(f"<td>{c}</td>" for c in row) + "</tr>"
    html += "</table>"
    display(HTML(html))


def show_fields(d: dict, keys: list[str] | None = None) -> None:
    keys = keys or list(d.keys())
    html = "<dl>"
    for k in keys:
        v = d.get(k)
        if isinstance(v, list):
            v = ", ".join(map(str, v)) if v else "\u2014"
        html += f"<dt><b>{k}</b></dt><dd>{v}</dd>"
    html += "</dl>"
    display(HTML(html))


LEVEL_ORDER = ["remember", "understand", "apply", "analyze", "evaluate", "create"]
LEVEL_COLORS = {
    "remember": "#64748b",
    "understand": "#3b82f6",
    "apply": "#10b981",
    "analyze": "#f59e0b",
    "evaluate": "#f97316",
    "create": "#8b5cf6",
}

RUBRIC_CRITERIA_ORDER = ["correctness", "completeness", "clarity", "readyToTeach"]
RUBRIC_CRITERIA_LABELS = ["Correctness", "Completeness", "Clarity", "Ready-to-teach"]
RUBRIC_CRITERIA_COLORS = {
    "correctness": "#3b82f6",
    "completeness": "#10b981",
    "clarity": "#f59e0b",
    "readyToTeach": "#8b5cf6",
}





def configure_publication_style() -> None:
    """Journal-friendly rcParams (serif, 300 dpi exports, minimal chart junk)."""
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.0)
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": ["Times New Roman", "DejaVu Serif", "Times"],
            "font.size": 10,
            "axes.labelsize": 10,
            "axes.titlesize": 10,
            "legend.fontsize": 8,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "figure.dpi": 120,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.alpha": 0.35,
            "grid.linewidth": 0.6,
        }
    )


def collect_score_progress_snapshots(initial_evaluation: dict, improve_result: dict | None = None) -> list[dict]:
    snapshots = [{"label": "Initial", "evaluation": initial_evaluation}]
    for round_result in (improve_result or {}).get("improve_history") or []:
        if round_result.get("status") == "completed" and round_result.get("evaluation"):
            snapshots.append(
                {
                    "label": f"Round {round_result.get('rounds_used', len(snapshots))}",
                    "evaluation": round_result["evaluation"],
                }
            )
    return snapshots


def plot_and_save_score_progress_figure(
    snapshots: list[dict],
    *,
    lecture_title: str | None = None,
    save_stem: Path | str | None = None,
    show_inline: bool = True,
) -> Path | None:
    """Two-panel figure: (a) alignment score trajectory, (b) stacked rubric score distribution.

    Saves vector PDF/SVG and 300 dpi PNG when save_stem is set (journal submission ready).
    """
    if not snapshots:
        return None

    configure_publication_style()

    labels = [snap["label"] for snap in snapshots]
    scores = [snap["evaluation"].get("alignmentScore", 0) for snap in snapshots]
    has_distribution = any((snap.get("evaluation") or {}).get("scores") for snap in snapshots)
    x = list(range(len(labels)))

    if has_distribution:
        fig, (ax_score, ax_dist) = plt.subplots(1, 2, figsize=(7.08, 2.75))
        panel_labels = ("(a)", "(b)")
    else:
        fig, ax_score = plt.subplots(figsize=(3.54, 2.75))
        ax_dist = None
        panel_labels = ("(a)",)

    ax_score.plot(x, scores, color="#5b21b6", linewidth=1.8, marker="o", markersize=5.5, zorder=3)
    for idx, score in enumerate(scores):
        ax_score.annotate(
            str(score),
            (x[idx], score),
            textcoords="offset points",
            xytext=(0, 7),
            ha="center",
            fontsize=8,
        )

    if ax_dist is not None:
        for key, label_c in zip(RUBRIC_CRITERIA_ORDER, RUBRIC_CRITERIA_LABELS):
            values = [
                (snap.get("evaluation") or {}).get("scores", {}).get(key, 0)
                for snap in snapshots
            ]
            ax_dist.plot(
                x, values,
                color=RUBRIC_CRITERIA_COLORS[key],
                linewidth=1.5,
                marker="o",
                markersize=4,
                label=label_c,
            )
            for idx, val in enumerate(values):
                if val:
                    ax_dist.annotate(
                        str(val), (x[idx], val),
                        textcoords="offset points", xytext=(0, 5),
                        ha="center", fontsize=7,
                    )
        ax_dist.set_ylim(0.5, 5.5)
        ax_dist.set_yticks([1, 2, 3, 4, 5])
        ax_dist.set_xticks(x)
        ax_dist.set_xticklabels(labels, rotation=15 if len(labels) > 4 else 0)
        ax_dist.set_ylabel("Score (1\u20135)")
        ax_dist.set_xlabel("Improvement stage")
        ax_dist.set_title("Rubric criteria scores", loc="left", fontweight="semibold")
        ax_dist.text(-0.12, 1.04, panel_labels[1], transform=ax_dist.transAxes, fontweight="bold")
        ax_dist.legend(loc="lower right", frameon=False)

    if lecture_title:
        fig.suptitle(lecture_title, fontsize=11, y=1.03)

    fig.tight_layout()

    saved: Path | None = None
    if save_stem is not None:
        saved = Path(str(save_stem))
        saved.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(saved.with_suffix(".pdf"))
        fig.savefig(saved.with_suffix(".png"), dpi=300)
        fig.savefig(saved.with_suffix(".svg"))
        print(f"Saved score progress figure: {saved.with_suffix('.pdf')} (+ .png 300dpi, .svg)")

    if show_inline:
        display(fig)
    plt.close(fig)
    return saved


def show_score_progress_charts(
    initial_evaluation: dict,
    improve_result: dict | None = None,
    *,
    lecture_title: str | None = None,
    save_stem: Path | str | None = None,
) -> Path | None:
    snapshots = collect_score_progress_snapshots(initial_evaluation, improve_result)
    if not snapshots:
        return None
    show_title("Score progress", level=4)
    return plot_and_save_score_progress_figure(
        snapshots,
        lecture_title=lecture_title,
        save_stem=save_stem,
        show_inline=True,
    )


def save_slide_eval_preview_html(
    deck: dict,
    evaluation: dict,
    lecture_title: str,
    path: Path,
) -> Path:
    """Human-readable HTML preview: every slide with its tag, blocks and speaker notes."""
    from html import escape

    slides = deck.get("slides") or []
    slide_tags = {}  # rubric eval has no per-slide tags

    def block_html(block: dict) -> str:
        btype = str(block.get("type", "block"))
        content = block.get("content")
        if isinstance(content, str):
            body = f"<p>{escape(content)}</p>"
        elif isinstance(content, dict):
            if isinstance(content.get("items"), list):
                body = "<ul>" + "".join(f"<li>{escape(str(i))}</li>" for i in content["items"]) + "</ul>"
            elif btype == "activity":
                body = "".join(
                    f"<p><strong>{escape(label)}:</strong> {escape(str(content[key]))}</p>"
                    for key, label in [("title", "Activity"), ("instructions", "Instructions"), ("purpose", "Purpose")]
                    if content.get(key)
                )
            elif content.get("prompt"):
                body = f"<p><strong>Prompt:</strong> {escape(str(content['prompt']))}</p>"
            elif content.get("code"):
                body = f"<pre>{escape(str(content['code']))}</pre>"
            else:
                body = f"<pre>{escape(json.dumps(content, indent=2, ensure_ascii=False))}</pre>"
        else:
            body = f"<pre>{escape(json.dumps(content, ensure_ascii=False))}</pre>"
        return (
            f'<div class="block"><div class="block-label">{escape(btype)}</div>{body}</div>'
        )

    rubric_scores = evaluation.get("scores") or {}
    dist_rows = "".join(
        f"<tr><td>{escape(label)}</td><td><strong>{rubric_scores.get(key, '\u2014')}/5</strong></td>"
        "</tr>"
        for key, label in [
            ("correctness", "Correctness"),
            ("completeness", "Completeness"),
            ("clarity", "Clarity"),
            ("readyToTeach", "Ready-to-teach"),
        ]
    )
    cards = []
    for idx, slide in enumerate(slides, 1):
        tag = slide_tags.get((slide.get("title") or "").strip().lower(), {})
        level = tag.get("primaryLevel", "untagged")
        color = LEVEL_COLORS.get(level, "#71717a")
        blocks = "".join(block_html(b) for b in slide.get("blocks") or [])
        notes = escape(slide.get("speakerNotes") or "")
        cards.append(
            f"""<article class="card">
<header><div><div class="kicker">Slide {idx:02d} · {escape(str(slide.get('layout', '')))}</div>
<h2>{escape(slide.get('title', 'Untitled'))}</h2></div>
</header>
{blocks}
<details open><summary>Speaker notes</summary><p>{notes}</p></details>
</article>"""
        )

    html = f"""<!doctype html><html lang="en"><head><meta charset="utf-8" />
<title>{escape(lecture_title)} — Rubric preview</title>
<style>
body{{margin:0;font:14px/1.55 -apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;color:#111827;background:#f8fafc}}
.page{{max-width:1100px;margin:0 auto;padding:32px 24px 80px}}
.hero,.card,.panel{{background:#fff;border:1px solid #e5e7eb;border-radius:16px;padding:22px;margin-bottom:18px}}
h1{{margin:0 0 6px;font-size:28px}} h2{{margin:2px 0 0;font-size:20px}}
.muted{{color:#6b7280}}
.metrics{{display:flex;gap:24px;margin-top:14px}}
.metrics b{{font-size:24px;display:block}}
table{{border-collapse:collapse}} td{{padding:6px 10px;border-bottom:1px solid #e5e7eb}}
.bar{{height:8px;width:140px;background:#e5e7eb;border-radius:99px;overflow:hidden}}
.bar span{{display:block;height:100%;border-radius:99px}}
.card header{{display:flex;justify-content:space-between;gap:16px;border-bottom:1px solid #e5e7eb;padding-bottom:12px;margin-bottom:12px}}
.kicker{{color:#6b7280;font-size:12px;text-transform:uppercase;letter-spacing:.08em}}
.level{{border:1px solid;border-radius:99px;padding:5px 10px;font-weight:700;font-size:12px;white-space:nowrap;align-self:flex-start}}
.evidence{{padding:10px 12px;border-left:4px solid #8b5cf6;background:#f5f3ff;border-radius:8px;margin:10px 0}}
.block{{border:1px solid #e5e7eb;border-radius:10px;padding:10px 12px;margin:8px 0}}
.block-label{{color:#6b7280;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;margin-bottom:4px}}
pre{{white-space:pre-wrap;background:#0f172a;color:#e5e7eb;padding:10px;border-radius:8px;overflow:auto}}
ul{{margin:4px 0 0 20px;padding:0}} p{{margin:6px 0}}
details{{margin-top:10px;color:#374151}} summary{{cursor:pointer;color:#6b7280;font-weight:700}}
@media print{{body{{background:#fff}}.card{{break-inside:avoid}}}}
</style></head><body><div class="page">
<section class="hero"><h1>{escape(lecture_title)}</h1>
<p class="muted">Slide-by-slide rubric preview · generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}</p>
<div class="metrics">
<div><span class="muted">Quality score</span><b>{evaluation.get('alignmentScore', '—')}</b></div>
<div><span class="muted">Slides</span><b>{len(slides)}</b></div>
</div></section>
<section class="panel"><h2>Score distribution</h2><table>{dist_rows}</table></section>
{''.join(cards)}
</div></body></html>"""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")
    print(f"Saved slide preview: {path}")
    return path


In [5]:
import io, math

# Style tuple: (bg_rgb, accent_rgb, text_rgb, mono, italic, font_size_pt)
# "columns" is handled specially as side-by-side layout — not in this table.
BLOCK_STYLES = {
    "code":       ((0x0f,0x17,0x2a), (0x10,0xb9,0x81), (0xd1,0xfa,0xe5), True,  False, 9),
    "diagram":    ((0xed,0xe9,0xfe), (0x6d,0x28,0xd9), (0x4c,0x1d,0x95), False, True,  10),
    "callout":    ((0xf0,0xf9,0xff), (0x0e,0xa5,0xe9), (0x07,0x5b,0x85), False, True,  10),
    "definition": ((0xf0,0xfd,0xf4), (0x16,0xa3,0x4a), (0x14,0x53,0x2d), False, False, 11),
    "activity":   ((0xff,0xf7,0xed), (0xea,0x58,0x0c), (0x7c,0x2d,0x12), False, False, 11),
    "table":      ((0xf8,0xfa,0xfc), (0x94,0xa3,0xb8), (0x1e,0x29,0x3b), True,  False, 9),
    "image":      ((0xf8,0xfa,0xfc), (0x94,0xa3,0xb8), (0x4b,0x55,0x63), False, True,  10),
}


def pdf_safe(text: str) -> str:
    return text.encode("latin-1", errors="replace").decode("latin-1")


def _col_body(col: dict) -> str:
    return re.sub(r"```[^\n]*\n?", "", col.get("content") or "").strip()


def _block_text(block: dict, bullet_char: str = "•") -> str:
    content = block.get("content")
    btype = block.get("type", "")
    if isinstance(content, str):
        return content
    if not isinstance(content, dict):
        return ""
    if btype == "bullets":
        return "\n".join(f"{bullet_char} {item}" for item in (content.get("items") or []))
    if btype == "code":
        return content.get("code") or ""
    if btype in ("diagram", "image"):
        return content.get("mermaid") or content.get("prompt") or content.get("alt") or ""
    if btype == "callout":
        return content.get("text") or ""
    if btype == "definition":
        term, defn = content.get("term") or "", content.get("definition") or ""
        return f"{term}: {defn}" if term else defn
    if btype == "activity":
        parts = ([f"[Activity] {content['title']}"] if content.get("title") else [])
        if content.get("duration"):
            parts.append(f"Duration: {content['duration']}")
        if content.get("instructions"):
            parts.append(content["instructions"])
        return "\n".join(parts)
    if btype == "table":
        h = content.get("headers") or []
        lines = (["  ".join(str(x) for x in h),
                  "  ".join("-" * max(len(str(x)), 4) for x in h)] if h else [])
        lines += ["  ".join(str(c) for c in r) for r in (content.get("rows") or [])]
        return "\n".join(lines)
    if btype == "columns":
        parts = []
        for col in content.get("columns") or []:
            hd = col.get("heading") or ""
            parts.append(f"{hd}\n{_col_body(col)}" if hd else _col_body(col))
        return "\n\n".join(parts)
    return ""


# ── PDF ───────────────────────────────────────────────────────────────────────

_LABEL_H = 16   # pt — space reserved for the block-type label row


def _pdf_n_lines(pdf, text: str, width: float) -> int:
    """Count visual lines after word-wrap, using current font metrics."""
    safe = pdf_safe(text)   # get_string_width uses the same latin-1 encoding
    return sum(
        max(1, math.ceil(pdf.get_string_width(line) / width)) if line.strip() else 1
        for line in safe.split("\n")
    )


def _render_boxed_block_pdf(pdf, btype: str, text: str) -> None:
    bg, accent, text_color, mono, italic, fs = BLOCK_STYLES[btype]
    font_family = "Courier" if mono else "Helvetica"
    font_style  = "I" if italic else ""
    line_h      = max(fs + 4, 12)
    text_w      = pdf.epw - 20          # 10 pt padding each side

    # Measure first so the pre-drawn rect matches the text height exactly
    pdf.set_font(font_family, style=font_style, size=fs)
    n_lines = _pdf_n_lines(pdf, text, text_w)
    box_h   = _LABEL_H + n_lines * line_h + 10   # 10 pt bottom padding

    x, y = pdf.get_x(), pdf.get_y()
    pdf.set_fill_color(*bg);     pdf.rect(x, y, pdf.epw, box_h, style="F")
    pdf.set_fill_color(*accent); pdf.rect(x, y, 3,       box_h, style="F")

    pdf.set_font("Helvetica", style="B", size=7)
    pdf.set_text_color(*accent)
    pdf.set_xy(x + 10, y + 5)
    pdf.cell(0, 9, btype.upper(), new_x="LMARGIN", new_y="NEXT")

    pdf.set_font(font_family, style=font_style, size=fs)
    pdf.set_text_color(*text_color)
    pdf.set_xy(x + 10, y + _LABEL_H)
    pdf.multi_cell(text_w, line_h, pdf_safe(text), new_x="LMARGIN", new_y="NEXT")


def _render_columns_pdf(pdf, columns: list) -> None:
    n       = len(columns)
    gap     = 8                              # pt between columns
    col_w   = (pdf.epw - gap * (n - 1)) / n
    start_x = pdf.get_x()
    start_y = pdf.get_y()
    end_ys: list[float] = []

    for i, col in enumerate(columns):
        heading = col.get("heading") or ""
        body    = _col_body(col)
        x       = start_x + i * (col_w + gap)
        y       = start_y

        if heading:
            pdf.set_font("Helvetica", style="B", size=10)
            pdf.set_text_color(0x1e, 0x29, 0x3b)
            pdf.set_xy(x, y)
            pdf.multi_cell(col_w, 14, pdf_safe(heading), new_x="LMARGIN", new_y="NEXT")
            y = pdf.get_y()

        if body:
            pdf.set_font("Helvetica", size=10)
            pdf.set_text_color(0x33, 0x41, 0x55)
            pdf.set_xy(x, y)
            pdf.multi_cell(col_w, 14, pdf_safe(body), new_x="LMARGIN", new_y="NEXT")
            y = pdf.get_y()

        end_ys.append(y)

    max_y = max(end_ys) if end_ys else start_y
    pdf.set_draw_color(0xd1, 0xd5, 0xdb)
    for i in range(n - 1):
        sep_x = start_x + i * (col_w + gap) + col_w + gap / 2
        pdf.line(sep_x, start_y, sep_x, max_y)

    if end_ys:
        pdf.set_y(max_y)


def export_deck_to_pdf(title: str, slides: list) -> bytes:
    pdf = FPDF(unit="pt", format="LETTER")
    pdf.set_auto_page_break(auto=True, margin=48)
    n = len(slides)
    for i, slide in enumerate(slides):
        pdf.add_page()
        pdf.set_font("Helvetica", size=9)
        pdf.set_text_color(0x7c, 0x3a, 0xed)
        pdf.cell(0, 12, pdf_safe(f"Slide {i + 1} of {n}"), align="R",
                 new_x="LMARGIN", new_y="NEXT")
        pdf.ln(4)
        pdf.set_font("Helvetica", style="B", size=24)
        pdf.set_text_color(0x1e, 0x29, 0x3b)
        pdf.multi_cell(0, 30, pdf_safe(slide.get("title", "")),
                       new_x="LMARGIN", new_y="NEXT")
        pdf.ln(8)
        for block in slide.get("blocks") or []:
            btype = block.get("type")
            if btype == "heading":
                continue
            if btype == "columns":
                cols = (block.get("content") or {}).get("columns") or []
                if cols:
                    _render_columns_pdf(pdf, cols)
                    pdf.ln(8)
                continue
            text = _block_text(block, bullet_char="-")
            if not text:
                continue
            if btype in BLOCK_STYLES:
                _render_boxed_block_pdf(pdf, btype, text)
            else:
                pdf.set_font("Helvetica", size=12)
                pdf.set_text_color(0x33, 0x41, 0x55)
                pdf.multi_cell(0, 16, pdf_safe(text), new_x="LMARGIN", new_y="NEXT")
            pdf.ln(8)
    return bytes(pdf.output())


# ── PPTX ─────────────────────────────────────────────────────────────────────

def _pptx_box_h(text: str, font_size: int):
    return max(Pt(len(text.split("\n")) * (font_size + 5) + 28), Pt(56))


def _render_boxed_block_pptx(sl, btype: str, text: str, y):
    bg, accent, text_color, mono, italic, fs = BLOCK_STYLES[btype]
    box_h = _pptx_box_h(text, fs)

    tb = sl.shapes.add_textbox(Inches(0.5), y, Inches(9), box_h)
    tb.fill.solid()
    tb.fill.fore_color.rgb = RGBColor(*bg)
    tf = tb.text_frame
    tf.word_wrap = True

    p_label = tf.paragraphs[0]
    p_label.text = btype.upper()
    p_label.font.size = Pt(7)
    p_label.font.bold = True
    p_label.font.color.rgb = RGBColor(*accent)

    p_text = tf.add_paragraph()
    p_text.text = text
    p_text.font.size = Pt(fs)
    p_text.font.italic = italic
    if mono:
        p_text.font.name = "Courier New"
    p_text.font.color.rgb = RGBColor(*text_color)

    return y + box_h + Pt(8)


def _render_columns_pptx(sl, columns: list, y):
    n       = len(columns)
    gap     = Pt(8)
    col_w   = (Inches(9) - gap * (n - 1)) / n
    start_x = Inches(0.5)
    max_y   = y

    for i, col in enumerate(columns):
        heading = col.get("heading") or ""
        body    = _col_body(col)
        x       = start_x + i * (col_w + gap)
        col_y   = y

        if heading:
            h_box = max(Pt(len(heading.split("\n")) * 16 + 8), Pt(24))
            tb = sl.shapes.add_textbox(x, col_y, col_w, h_box)
            tf = tb.text_frame
            tf.word_wrap = True
            p = tf.paragraphs[0]
            p.text = heading
            p.font.size = Pt(12)
            p.font.bold = True
            p.font.color.rgb = RGBColor(0x1e, 0x29, 0x3b)
            col_y += h_box

        if body:
            body_h = max(Pt(len(body.split("\n")) * 15 + 8), Pt(40))
            tb = sl.shapes.add_textbox(x, col_y, col_w, body_h)
            tf = tb.text_frame
            tf.word_wrap = True
            p = tf.paragraphs[0]
            p.text = body
            p.font.size = Pt(11)
            p.font.color.rgb = RGBColor(0x33, 0x41, 0x55)
            col_y += body_h

        max_y = max(max_y, col_y)

    return max_y + Pt(8)


def export_deck_to_pptx(title: str, slides: list) -> bytes:
    prs = Presentation()
    prs.core_properties.author = "ProfDeck"
    prs.core_properties.title = title
    prs.slide_width = Inches(10)
    prs.slide_height = Inches(7.5)
    blank_layout = prs.slide_layouts[6]
    for slide_data in slides:
        sl = prs.slides.add_slide(blank_layout)
        tb = sl.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(9), Inches(0.8))
        tf = tb.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.text = slide_data.get("title", "")
        p.font.size = Pt(28)
        p.font.bold = True
        p.font.color.rgb = RGBColor(0x1e, 0x29, 0x3b)
        y = Inches(1.2)
        for block in slide_data.get("blocks") or []:
            btype = block.get("type")
            if btype == "heading":
                continue
            if btype == "columns":
                cols = (block.get("content") or {}).get("columns") or []
                if cols:
                    y = _render_columns_pptx(sl, cols, y)
                continue
            text = _block_text(block)
            if not text:
                continue
            if btype in BLOCK_STYLES:
                y = _render_boxed_block_pptx(sl, btype, text, y)
            else:
                tb = sl.shapes.add_textbox(Inches(0.5), y, Inches(9), Inches(1.5))
                tf = tb.text_frame
                tf.word_wrap = True
                p = tf.paragraphs[0]
                p.text = text
                p.font.size = Pt(16 if btype == "bullets" else 14)
                p.font.color.rgb = RGBColor(0x33, 0x41, 0x55)
                y += Inches(1.6)
    buf = io.BytesIO()
    prs.save(buf)
    return buf.getvalue()


## Step 0.5 — Lecture plan (extract or generate)

Builds a **course-level lecture list** (target **10–12 lectures**) for the selected handbook module.

1. Try to **extract** lectures from handbook text (`Lecture 1`, `Week 2`, numbered sessions, …).
2. If fewer than `MIN_LECTURES_FROM_HANDBOOK` are found, **generate** a lecture plan with AI from `COURSE_CONTEXT`.
3. Save artifacts under `output/course-generation/`:
   - `{module-code}-lecture-plan.json`
   - `{module-code}-lecture-plan.md`
   - ~~PDF report~~ **disabled for now** (see `GENERATE_PDF_REPORT` in Step 4)

Edit `TARGET_LECTURE_COUNT` (10–12) in the next cell if needed.

In [6]:
from datetime import datetime, timezone

# --- Lecture plan settings (edit these) ---
TARGET_LECTURE_COUNT = 10  # use 10–12
MIN_LECTURES_FROM_HANDBOOK = 8  # if extracted count is below this, generate instead
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "course-generation"

LECTURE_PLAN_SYSTEM_PROMPT = """You are an expert CS curriculum designer.
Design a university module delivery plan as a list of lectures.
Output ONLY valid JSON."""

LECTURE_PLAN_USER_PROMPT = """Design a lecture plan for this handbook module.

Module title: {module_title}
Module code: {module_code}
Target number of lectures: {target_count} (each about 90 minutes)

Handbook context:
{handbook_context}

Return JSON:
{{
  "source": "generated",
  "targetLectureCount": {target_count},
  "lectures": [
    {{
      "number": number,
      "title": string,
      "durationMinutes": 90,
      "summary": string,
      "topics": string[]
    }}
  ]
}}

Rules:
- Produce exactly {target_count} lectures in logical teaching order
- Base topics on module description and intended learning outcomes from the handbook
- Titles should be lecture-ready (clear, specific, MSc Software Engineering level)
- No duplicate or overlapping lecture titles"""




def module_slug(module_code: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "-", module_code).strip("-").lower()


def extract_module_text(section_text: str, module_code: str, module_name: str) -> str:
    """Narrow handbook text to the selected module block when possible."""
    pattern = rf"Module Name\s+{re.escape(module_name)}[\s\S]*?(?=Module Name\s+|\Z)"
    match = re.search(pattern, section_text, flags=re.IGNORECASE)
    if match:
        return match.group(0)
    if module_code in section_text:
        start = section_text.find(module_code)
        return section_text[start : start + 8000]
    return section_text


def extract_lectures_from_text(text: str) -> list[dict]:
    """Heuristic extraction of lecture/session lists from handbook-like text."""
    lectures: list[dict] = []
    seen_numbers: set[int] = set()

    labeled = re.finditer(
        r"(?:^|\n)\s*(?:Lecture|Session|Week|Unit)\s+(\d{1,2})\s*[\.:\-–—]\s*(.+?)(?=(?:\n\s*(?:Lecture|Session|Week|Unit)\s+\d{1,2}\s*[\.:\-–—])|\Z)",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    for match in labeled:
        number = int(match.group(1))
        if number in seen_numbers:
            continue
        title = re.sub(r"\s+", " ", match.group(2)).strip(" .")
        title = re.split(r"\n\s*(?:Duration|Target|Primary goal|Module|Assessment)\b", title, maxsplit=1)[0].strip()
        if len(title) < 8:
            continue
        seen_numbers.add(number)
        lectures.append(
            {
                "number": number,
                "title": title,
                "durationMinutes": 90,
                "summary": "",
                "topics": [],
            }
        )

    if len(lectures) < MIN_LECTURES_FROM_HANDBOOK:
        numbered = re.finditer(r"(?:^|\n)\s*(\d{1,2})\.\s+([A-Z][^\n]{12,140})", text)
        for match in numbered:
            number = int(match.group(1))
            if number in seen_numbers or number > 20:
                continue
            title = re.sub(r"\s+", " ", match.group(2)).strip()
            if any(skip in title.lower() for skip in ("module", "assessment", "ects", "cp")):
                continue
            seen_numbers.add(number)
            lectures.append(
                {
                    "number": number,
                    "title": title,
                    "durationMinutes": 90,
                    "summary": "",
                    "topics": [],
                }
            )

    lectures.sort(key=lambda item: item["number"])
    return lectures


def generate_lecture_plan(module: dict, handbook_context: str, target_count: int) -> dict:
    print(f"=== Step 0.5: generating {target_count} lectures for '{module['name']}' ===")

    def _call():
        user = LECTURE_PLAN_USER_PROMPT.format(
            module_title=module["name"],
            module_code=module["code"],
            target_count=target_count,
            handbook_context=handbook_context[:12000],
        )
        return call_llm(LECTURE_PLAN_SYSTEM_PROMPT, user, temperature=0.4)

    plan = parse_with_retry(_call, label="lecture-plan")
    plan["source"] = "generated"
    plan["targetLectureCount"] = target_count
    return plan


def build_lecture_plan(selected_module: dict, course_context: str, handbook_section_text: str, target_count: int) -> dict:
    module_text = extract_module_text(
        handbook_section_text,
        selected_module["code"],
        selected_module["name"],
    )
    extracted = extract_lectures_from_text(module_text)
    if len(extracted) >= MIN_LECTURES_FROM_HANDBOOK:
        print(f"=== Step 0.5: extracted {len(extracted)} lectures from handbook text ===")
        return {
            "source": "handbook",
            "targetLectureCount": len(extracted),
            "moduleCode": selected_module["code"],
            "moduleName": selected_module["name"],
            "lectures": extracted,
        }

    generated = generate_lecture_plan(selected_module, course_context, target_count)
    generated["moduleCode"] = selected_module["code"]
    generated["moduleName"] = selected_module["name"]
    return generated


def save_lecture_plan_json(plan: dict, path: Path) -> None:
    path.write_text(json.dumps(plan, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def save_lecture_plan_markdown(plan: dict, path: Path) -> None:
    lines = [
        f"# Lecture plan — {plan.get('moduleName', 'Module')}",
        "",
        f"- **Module code:** {plan.get('moduleCode', '—')}",
        f"- **Source:** {plan.get('source', '—')}",
        f"- **Lecture count:** {len(plan.get('lectures', []))}",
        f"- **Generated at:** {datetime.now(timezone.utc).isoformat()}",
        "",
        "## Lectures",
        "",
    ]
    for lecture in plan.get("lectures", []):
        lines.append(f"### Lecture {lecture.get('number')}. {lecture.get('title')}")
        if lecture.get("durationMinutes"):
            lines.append(f"- **Duration:** {lecture['durationMinutes']} minutes")
        if lecture.get("summary"):
            lines.append(f"- **Summary:** {lecture['summary']}")
        topics = lecture.get("topics") or []
        if topics:
            lines.append("- **Topics:**")
            lines.extend(f"  - {topic}" for topic in topics)
        lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")


def persist_lecture_plan(plan: dict, course_title: str) -> dict[str, Path]:
    slug = module_slug(plan.get("moduleCode", "module"))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    paths = {
        "json": OUTPUT_DIR / f"{slug}-lecture-plan.json",
        "markdown": OUTPUT_DIR / f"{slug}-lecture-plan.md",
        "full_run_json": OUTPUT_DIR / f"{slug}-full-run.json",
        "full_run_markdown": OUTPUT_DIR / f"{slug}-full-run.md",
        "lectures_dir": OUTPUT_DIR / slug / "lectures",
    }
    save_lecture_plan_json(plan, paths["json"])
    save_lecture_plan_markdown(plan, paths["markdown"])
    return paths


# --- Run Step 0.5 (executes when you run this cell) ---
if not (10 <= TARGET_LECTURE_COUNT <= 12):
    raise ValueError("TARGET_LECTURE_COUNT must be between 10 and 12")

LECTURE_PLAN = build_lecture_plan(
    selected_module=selected_module,
    course_context=COURSE_CONTEXT,
    handbook_section_text=handbook_section_text,
    target_count=TARGET_LECTURE_COUNT,
)
LECTURE_PLAN_PATHS = persist_lecture_plan(LECTURE_PLAN, COURSE_TITLE)

show_title("Lecture plan", level=3)
show_fields(
    {
        "source": LECTURE_PLAN.get("source"),
        "lecture_count": len(LECTURE_PLAN.get("lectures", [])),
        "json": str(LECTURE_PLAN_PATHS["json"]),
        "markdown": str(LECTURE_PLAN_PATHS["markdown"]),
    }
)
show_table(
    [
        [lec.get("number"), lec.get("title"), lec.get("durationMinutes", 90)]
        for lec in LECTURE_PLAN.get("lectures", [])
    ],
    headers=["#", "Lecture title", "Minutes"],
)
print(f"Saved lecture plan ({LECTURE_PLAN['source']}) to {OUTPUT_DIR}")

=== Step 0.5: generating 10 lectures for 'Software Construction, Architecture and Engineering' ===
[lecture-plan] attempt 1/3
    -> calling anthropic/claude-sonnet-4.5 via openrouter (temperature=0.4, max_tokens=8192)...


    <- received response in 17.4s (5953 chars)
[lecture-plan] parsed valid JSON on attempt 1


#,Lecture title,Minutes
1,Introduction to Software Engineering and Object-Oriented Fundamentals,90
2,Advanced Object-Oriented Mechanisms: Inheritance and Polymorphism,90
3,Genericity and Design by Contract,90
4,Fundamental Design Patterns I: Creational and Structural Patterns,90
5,Fundamental Design Patterns II: Behavioral Patterns,90
6,Software Architecture: Structuring Large-Scale Systems,90
7,Requirements Engineering Fundamentals,90
8,Agile Development Methods and Practices,90
9,Software Quality and Testing Strategies,90
10,Software Engineering Process and Project Management,90


Saved lecture plan (generated) to /home/ilgiz/sci/lectura/output/course-generation


## Steps 1–3 — Generators (outline, slides, rubric eval)

Prompt templates and `generate_*` functions used by **Step 4** for each lecture in `LECTURE_PLAN`.
The old single-lecture demo runs below are disabled — use Step 4 instead.


In [7]:
OUTLINE_SYSTEM_PROMPT = """You are an expert computer science professor and curriculum designer.
Generate a structured university lecture outline in JSON format.
Focus on clear learning progression, practical examples, and CS pedagogy.
Design the outline for a slide deck of approximately {target_slides} slides (one main idea per slide).
Output ONLY valid JSON matching the requested schema."""

OUTLINE_USER_PROMPT = """Create a lecture outline with the following parameters:
Title: {title}
Description: {description}
Target slide count: {target_slides}

Return JSON with this structure:
{{
  "title": string,
  "summary": string,
  "learningObjectives": string[],
  "targetSlideCount": {target_slides},
  "estimatedStudyMinutes": number,
  "sections": [
    {{
      "title": string,
      "learningGoals": string[],
      "subsections": [
        {{
          "title": string,
          "keyPoints": string[],
          "examples": string[],
          "suggestedDiagram": string
        }}
      ],
      "exercises": string[]
    }}
  ],
  "suggestedResources": string[],
  "assessmentIdeas": string[]
}}"""


def generate_outline(title: str, description: str = "", target_slides: int = 10) -> dict:
    print(f"=== Step 1: generating outline for \'{title}\' ===")

    def _call():
        system = OUTLINE_SYSTEM_PROMPT.format(target_slides=target_slides)
        user = OUTLINE_USER_PROMPT.format(
            title=title, description=description or "N/A", target_slides=target_slides
        )
        return call_llm(system, user, temperature=0.5)

    return parse_with_retry(_call, label="outline")


In [8]:
# Single-lecture demo disabled.
# Run Step 4 to iterate outline -> slides -> eval for every lecture in LECTURE_PLAN.
print("Step 1 demo skipped. Use Step 4.")


Step 1 demo skipped. Use Step 4.


## Step 2 — Slide deck generation

Takes the outline from Step 1 and turns it into a slide deck.

In [9]:
SLIDES_SYSTEM_PROMPT = """You are an expert presentation designer for university CS lectures.
Convert lecture outlines into engaging slide decks with speaker notes.
Use concise bullet points (max 5 per slide), include code snippets when relevant.
Output ONLY valid JSON."""

SLIDES_USER_PROMPT = """Convert this lecture outline into a slide deck for "{lecture_title}".
Target {target_slides} slides (\u00b12 if needed for pacing).

Each slide must map to a lecture section. Follow cognitive flow: conceptual -> visual -> code -> recap.
Avoid text-heavy slides.

Outline:
{outline_json}

Return JSON:
{{
  "slides": [
    {{
      "id": string,
      "title": string,
      "layout": "title" | "content" | "split" | "code" | "image",
      "blocks": [
        {{
          "id": string,
          "type": "heading" | "text" | "bullets" | "code" | "image" | "diagram" | "quote",
          "content": string | {{"items": string[], "language": string, "code": string, "prompt": string}}
        }}
      ],
      "speakerNotes": string,
      "metadata": {{"transition": string, "animation": string, "tags": string[]}}
    }}
  ]
}}

Guidelines:
- Start with a title slide
- One main idea per content slide
- Add diagram blocks with descriptive prompts where visuals help
- Include speaker notes for teaching tips"""


def generate_slides(outline: dict, lecture_title: str) -> dict:
    target_slides = outline.get("targetSlideCount", 10)
    print(f"=== Step 2: generating ~{target_slides} slides for \'{lecture_title}\' ===")

    def _call():
        user = SLIDES_USER_PROMPT.format(
            lecture_title=lecture_title,
            target_slides=target_slides,
            outline_json=json.dumps(outline, indent=2),
        )
        return call_llm(SLIDES_SYSTEM_PROMPT, user, temperature=0.6, max_tokens=8192)

    return parse_with_retry(_call, label="slides")


In [10]:
# Single-lecture demo disabled. See Step 4.
print("Step 2 demo skipped. Use Step 4.")


Step 2 demo skipped. Use Step 4.


## Step 3 — Rubric-based evaluation

Evaluates the generated deck against the outline's learning objectives, using the 4-criterion rubric (Correctness, Completeness, Clarity, Ready-to-teach).

In [11]:
RUBRIC_DOC = """## Criteria (1–5)

### 1. Correctness
Are facts, terminology, examples, and any code/diagrams accurate?

| Score | Meaning |
|------:|---------|
| 1 | Many serious errors; unsafe to show students. |
| 2 | Several important errors; major correction needed. |
| 3 | Mostly OK, but some clear mistakes / imprecise wording. |
| 4 | Accurate overall; only minor slips. |
| 5 | Technically sound and precise for an MSc audience. |

### 2. Completeness
Does the deck cover the lecture topic and learning objectives well enough for one session?

| Score | Meaning |
|------:|---------|
| 1 | Core topic largely missing. |
| 2 | Major gaps in important subtopics / objectives. |
| 3 | Main ideas present, but coverage is uneven. |
| 4 | Solid coverage; small gaps only. |
| 5 | Complete enough for the planned lecture. |

### 3. Clarity
Is the deck understandable and well-structured for MSc students?

| Score | Meaning |
|------:|---------|
| 1 | Confusing; hard to follow. |
| 2 | Often unclear; weak structure or overloaded slides. |
| 3 | Understandable with effort; some vague / dense parts. |
| 4 | Clear overall; only occasional issues. |
| 5 | Easy to follow; good pacing and structure. |

### 4. Ready-to-teach / Overall quality
How ready is this deck for real classroom use?

| Score | Meaning |
|------:|---------|
| 1 | Not ready; would need rewriting. |
| 2 | Far from ready; large edits needed. |
| 3 | Draft-ready; usable as a starting point. |
| 4 | Nearly ready; only light polishing. |
| 5 | Classroom-ready with at most tiny fixes. |

### Final question (required)
Would you use this lecture with students after light editing?
- Yes — ready enough, tiny polish only
- With edits — usable after moderate edits
- No — needs major rework"""

EVALUATION_SYSTEM_PROMPT = """You are an expert CS professor evaluating a university lecture slide deck.
Apply the rubric below exactly as a human evaluator would — use the same anchor descriptions.
Score each criterion as an integer 1–5. Output ONLY valid JSON."""

EVALUATION_USER_PROMPT = """Evaluate this lecture slide deck using the rubric below.

Slides:
{slides_json}

{rubric_doc}

Return JSON:
{{
  "scores": {{
    "correctness": <integer 1-5>,
    "completeness": <integer 1-5>,
    "clarity": <integer 1-5>,
    "readyToTeach": <integer 1-5>
  }},
  "useAfterEditing": "Yes" | "With edits" | "No"
}}

Rules:
- Use integers only (1, 2, 3, 4, or 5) — no decimals, no ranges
- Do not evaluate visual design or slide formatting"""


def evaluate_deck(lecture_title: str, slides: list) -> dict:
    print(f"=== Step 3: evaluating {len(slides)} slides against rubric ===")

    def _call():
        # Strip speaker notes — evaluator sees only what students see
        slides_clean = [
            {
                "title": slide["title"],
                "blocks": [
                    {"type": b["type"], "content": b.get("content", "")}
                    for b in slide.get("blocks", [])
                    if b.get("type", "") != "heading" and b.get("type")
                ],
            }
            for slide in slides
        ]
        user = EVALUATION_USER_PROMPT.format(
            slides_json=json.dumps(slides_clean, indent=2),
            rubric_doc=RUBRIC_DOC,
        )
        return call_llm(EVALUATION_SYSTEM_PROMPT, user, temperature=0.3)

    result = parse_with_retry(_call, label="evaluation")
    scores = result.get("scores") or {}
    total = sum(scores.get(k, 1) for k in ("correctness", "completeness", "clarity", "readyToTeach"))
    result["alignmentScore"] = round((total - 4) / 16 * 100)
    return result


## Step 3.5 — Improve & re-evaluate

In [12]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_IMPROVE_ROUNDS = 3
IMPROVE_MAX_WORKERS = 8

SINGLE_SLIDE_SYSTEM_PROMPT = """You are an expert CS lecture slide designer.
Regenerate a single slide while preserving context from the surrounding deck.
Output ONLY valid JSON for one slide object."""

SINGLE_SLIDE_USER_PROMPT = """Lecture: {lecture_title}

Outline summary:
{outline_summary}

Adjacent slides:
{adjacent_slides}

Current slide to regenerate:
{current_slide}

{instructions_block}

Return JSON for a single slide:
{{
  "id": string,
  "title": string,
  "layout": "title" | "content" | "split" | "code" | "image",
  "blocks": [{{ "id": string, "type": string, "content": string | object }}],
  "speakerNotes": string,
  "metadata": {{}}
}}

Size limits (STRICT — oversized responses get truncated and break JSON parsing):
- At most 4 blocks total; at most 1 activity block
- speakerNotes: max ~150 words (plain prose, no headings or timelines)
- Each activity explanation: max ~80 words
- Keep the whole JSON response under 4000 characters"""


def build_improvement_instructions(
    evaluation: dict,
    slide_title: str,
) -> str:
    scores = evaluation.get("scores") or {}
    criteria_labels = {
        "correctness": "Correctness",
        "completeness": "Completeness",
        "clarity": "Clarity",
        "readyToTeach": "Ready-to-teach",
    }
    lines = [
        "Improve this slide based on the rubric evaluation of the full deck.",
        "Only this slide was selected — focus changes here, keep title/topic.",
        "",
        f"Deck quality score: {evaluation.get('alignmentScore')}/100.",
        "",
        "Current rubric scores for the deck (1–5):",
    ]
    for key, label in criteria_labels.items():
        score = scores.get(key, "?")
        lines.append(f"- {label}: {score}/5")
    lines.append("")
    lines.append(
        f'Rewrite slide "{slide_title}" to improve correctness, completeness, clarity, and readiness.'
    )
    lines.append("Keep the slide topic and its place in the deck.")
    return "\n".join(lines)


def regenerate_single_slide(
    lecture_title: str,
    outline_summary: str,
    adjacent_slides: list,
    current_slide: dict,
    instructions: str | None = None,
) -> dict:
    instructions_block = f"Instructions: {instructions}" if instructions else ""

    def _call():
        user = SINGLE_SLIDE_USER_PROMPT.format(
            lecture_title=lecture_title,
            outline_summary=outline_summary,
            adjacent_slides=json.dumps(adjacent_slides, indent=2),
            current_slide=json.dumps(current_slide, indent=2),
            instructions_block=instructions_block,
        )
        return call_llm(SINGLE_SLIDE_SYSTEM_PROMPT, user, temperature=0.6)

    result = parse_with_retry(_call, label="single-slide")
    return {**result, "id": current_slide.get("id", result.get("id"))}


def improve_and_reevaluate(
    lecture_title: str,
    outline: dict,
    deck: dict,
    evaluation: dict,
    rounds_used: int,
    *,
    reevaluate: bool = True,
) -> dict:
    before_score = evaluation.get("alignmentScore", 0)
    slides = deck.get("slides") or []

    if rounds_used >= MAX_IMPROVE_ROUNDS:
        return {
            "status": "skipped",
            "skipped_reason": f"Improve limit reached ({MAX_IMPROVE_ROUNDS} rounds).",
            "deck": deck,
            "evaluation": evaluation,
            "rounds_used": rounds_used,
            "before_score": before_score,
            "after_score": before_score,
            "improved_slide_ids": [],
            "improved_count": 0,
            "reevaluated": False,
        }

    updated_by_id: dict[str, dict] = {}
    improved_slide_ids: list[str] = []
    failed_slide_ids: list[str] = []
    outline_summary = outline.get("summary", "")

    print(
        f"=== Step 3.5: improving {len(slides)} slide(s) "
        f"(round {rounds_used + 1}/{MAX_IMPROVE_ROUNDS}, score {before_score}) ==="
    )

    def _regen_one(args):
        i, current = args
        slide_id = current.get("id")
        adjacent = [slides[i - 1]] if i > 0 else []
        if i < len(slides) - 1:
            adjacent.append(slides[i + 1])
        instructions = build_improvement_instructions(evaluation=evaluation, slide_title=current.get("title", ""))
        try:
            regenerated = regenerate_single_slide(
                lecture_title=lecture_title,
                outline_summary=outline_summary,
                adjacent_slides=adjacent,
                current_slide=current,
                instructions=instructions,
            )
            return i, slide_id, regenerated, None
        except (RuntimeError, requests.RequestException) as exc:
            return i, slide_id, None, exc

    with ThreadPoolExecutor(max_workers=IMPROVE_MAX_WORKERS) as pool:
        futures = {pool.submit(_regen_one, (i, s)): i for i, s in enumerate(slides)}
        done_count = 0
        for fut in as_completed(futures):
            i, slide_id, regenerated, exc = fut.result()
            done_count += 1
            if exc is not None:
                failed_slide_ids.append(slide_id)
                print(f"  !! [{done_count}/{len(slides)}] keeping original (failed: {str(exc).splitlines()[0][:120]})")
            else:
                updated_by_id[slide_id] = regenerated
                improved_slide_ids.append(slide_id)
                print(f"  [{done_count}/{len(slides)}] done: {slides[i].get('title')}")

    if failed_slide_ids:
        print(
            f"  !! {len(failed_slide_ids)} slide(s) kept unchanged after failed regeneration: "
            f"{', '.join(str(s) for s in failed_slide_ids)}"
        )

    if not improved_slide_ids:
        return {
            "status": "skipped",
            "skipped_reason": "All slide regenerations failed this round; deck unchanged.",
            "deck": deck,
            "evaluation": evaluation,
            "rounds_used": rounds_used,
            "before_score": before_score,
            "after_score": before_score,
            "improved_slide_ids": [],
            "failed_slide_ids": failed_slide_ids,
            "improved_count": 0,
            "reevaluated": False,
        }

    slides = [updated_by_id.get(s.get("id"), s) for s in slides]
    deck = {**deck, "slides": slides}

    after_score = before_score
    reevaluated = False
    if reevaluate:
        print("=== Step 3.5: re-evaluating deck after improve ===")
        try:
            evaluation = evaluate_deck(
                lecture_title=lecture_title,
                slides=slides,
            )
            after_score = evaluation.get("alignmentScore", before_score)
            reevaluated = True
        except (RuntimeError, requests.RequestException) as exc:
            print(f"  !! re-evaluation failed, keeping previous scores: {str(exc).splitlines()[0][:160]}")

    return {
        "status": "completed",
        "deck": deck,
        "evaluation": evaluation,
        "rounds_used": rounds_used + 1,
        "before_score": before_score,
        "after_score": after_score,
        "improved_slide_ids": improved_slide_ids,
        "failed_slide_ids": failed_slide_ids,
        "improved_count": len(improved_slide_ids),
        "reevaluated": reevaluated,
    }


def should_revert_improve(before_score: int, after_score: int, reevaluated: bool) -> bool:
    return reevaluated and after_score < before_score


def run_improve_loop(
    lecture_title: str,
    outline: dict,
    deck: dict,
    evaluation: dict,
    *,
    auto_improve: bool = True,
    reevaluate: bool = True,
) -> dict:
    """Auto-loop up to MAX_IMPROVE_ROUNDS (batch equivalent of UI Improve clicks)."""
    if not auto_improve:
        return {
            "deck": deck,
            "evaluation": evaluation,
            "improve_history": [],
            "rounds_used": 0,
            "initial_score": evaluation.get("alignmentScore"),
        }

    improve_history = []
    rounds_used = 0
    initial_score = evaluation.get("alignmentScore")

    while rounds_used < MAX_IMPROVE_ROUNDS:
        if evaluation.get("alignmentScore", 0) >= 100:
            print(f"  -> score {evaluation.get('alignmentScore')}/100 — stopping improve")
            break
        pre_round_deck = deck
        pre_round_evaluation = evaluation
        result = improve_and_reevaluate(
            lecture_title=lecture_title,
            outline=outline,
            deck=deck,
            evaluation=evaluation,
            rounds_used=rounds_used,
            reevaluate=reevaluate,
        )
        improve_history.append(result)
        if result["status"] == "skipped":
            break
        rounds_used = result["rounds_used"]

        if should_revert_improve(
            result["before_score"],
            result["after_score"],
            result.get("reevaluated", False),
        ):
            deck = pre_round_deck
            evaluation = pre_round_evaluation
            result["kept_best"] = True
            result["raw_after_score"] = result["after_score"]
            result["after_score"] = result["before_score"]
            result["deck"] = deck
            result["evaluation"] = evaluation
            print(
                f"  -> round {rounds_used}: {result['before_score']} -> {result['raw_after_score']} "
                f"(regressed, kept best at {result['after_score']})"
            )
        else:
            deck = result["deck"]
            evaluation = result["evaluation"]
            print(
                f"  -> round {rounds_used}: {result['before_score']} -> {result['after_score']} "
                f"({result['improved_count']} slides)"
            )

    return {
        "deck": deck,
        "evaluation": evaluation,
        "improve_history": improve_history,
        "rounds_used": rounds_used,
        "initial_score": initial_score,
        "final_score": evaluation.get("alignmentScore"),
    }


## Step 4 — Run pipeline for all lectures

Iterates over `LECTURE_PLAN["lectures"]` and for **each lecture** runs:

1. outline generation
2. slide deck generation
3. Rubric evaluation
4. Improve loop (Step 3.5) — up to 3 rounds until score ≥ 90

Artifacts saved under `output/course-generation/{module}/lectures/`:
- per-lecture JSON: outline, deck, evaluation, improve-history
- per-lecture HTML slide preview
- per-lecture **Score progress figure** (PDF/PNG/SVG) for papers
- `{module}-full-run.json` and `{module}-full-run.md`

PDF and PPTX exports are handled by **Step 5**.

Default: **smoke test lecture 1**, then auto-continue with the rest (`SMOKE_TEST_FIRST = True`).
Set `MAX_LECTURES_TO_PROCESS = 1` to stop after smoke (no auto-continue).
Set `SMOKE_TEST_FIRST = False` to process all lectures in one pass.
Set `AUTO_IMPROVE = False` to skip improve (evaluate only).

In [13]:
# --- Step 4 settings ---
SLIDES_PER_LECTURE = 18
SMOKE_TEST_FIRST = True       # run lecture 1 first; on success auto-continue
MAX_LECTURES_TO_PROCESS = None  # None = full course; set 1 for smoke-only
AUTO_IMPROVE = True
REEVALUATE_AFTER_IMPROVE = True
SKIP_COMPLETED = True  # skip lectures whose deck JSON already exists on disk


def build_lecture_description(lecture: dict, module: dict, course_context: str) -> str:
    topics = lecture.get("topics") or []
    lines = [
        f"Module: {module['name']} ({module['code']})",
        f"Lecture number: {lecture.get('number')}",
        f"Lecture title: {lecture.get('title')}",
        f"Duration: {lecture.get('durationMinutes', 90)} minutes",
    ]
    if lecture.get("summary"):
        lines.append(f"Lecture summary: {lecture['summary']}")
    if topics:
        lines.append("Lecture topics: " + "; ".join(topics))
    lines.extend(["", "Handbook / module context:", course_context[:8000]])
    return "\n".join(lines)


def lecture_artifact_paths(base_dir: Path, lecture_number: int) -> dict[str, Path]:
    prefix = base_dir / f"lecture-{lecture_number:02d}"
    return {
        "outline": Path(f"{prefix}-outline.json"),
        "deck": Path(f"{prefix}-deck.json"),
        "evaluation": Path(f"{prefix}-evaluation.json"),
        "improve_history": Path(f"{prefix}-improve-history.json"),
        "score_progress_figure": Path(f"{prefix}-score-progress"),
        "preview": Path(f"{prefix}-slide-preview.html"),
        "pdf": Path(f"{prefix}-slides.pdf"),
        "pptx": Path(f"{prefix}-slides.pptx"),
    }


def save_full_run_markdown(run_bundle: dict, path: Path) -> None:
    plan = run_bundle["lecturePlan"]
    lines = [
        f"# Full run — {plan.get('moduleName')}",
        "",
        f"- **Module code:** {plan.get('moduleCode')}",
        f"- **Lectures processed:** {len(run_bundle['lectures'])}",
        f"- **Generated at:** {run_bundle['generatedAt']}",
        "",
        "| # | Lecture | Score init | Score final | Rounds |",
        "|---|---------|------------|-------------|--------|",
    ]
    for item in run_bundle["lectures"]:
        ev = item.get("evaluation") or {}
        lines.append(
            f"| {item['number']} | {item['title']} | {item.get('initialScore', '—')} | "
            f"{ev.get('alignmentScore', '—')} | {item.get('improveRoundsUsed', 0)} |"
        )
    lines.append("")
    for item in run_bundle["lectures"]:
        lines.append(f"## Lecture {item['number']}. {item['title']}")
        ev = item.get("evaluation") or {}
        outline = item.get("outline") or {}
        lines.append(f"- **Outline summary:** {outline.get('summary', '—')}")
        lines.append(f"- **Score:** {item.get('initialScore', '—')} → {ev.get('alignmentScore', '—')}")
        gaps = ev.get("gaps") or []
        if gaps:
            lines.append("- **Gaps:**")
            lines.extend(f"  - {g}" for g in gaps[:3])
        lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")


def run_pipeline_for_all_lectures(
    lecture_plan: dict,
    module: dict,
    course_context: str,
    course_title: str,
    paths: dict[str, Path],
    *,
    slides_per_lecture: int = SLIDES_PER_LECTURE,
    max_lectures: int | None = MAX_LECTURES_TO_PROCESS,
    smoke_test_first: bool = SMOKE_TEST_FIRST,
    auto_improve: bool = AUTO_IMPROVE,
    reevaluate_after_improve: bool = REEVALUATE_AFTER_IMPROVE,
    skip_completed: bool = SKIP_COMPLETED,
) -> dict:
    plan_lectures = lecture_plan.get("lectures", [])
    if max_lectures is not None:
        plan_lectures = plan_lectures[:max_lectures]

    lectures_dir = paths["lectures_dir"]
    lectures_dir.mkdir(parents=True, exist_ok=True)
    run_items: list[dict] = []
    total = len(plan_lectures)

    def process_lectures(batch: list[dict], phase_label: str) -> None:
        nonlocal run_items
        print(phase_label)
        for lecture in batch:
            number = int(lecture.get("number") or len(run_items) + 1)
            title = lecture.get("title") or f"Lecture {number}"
            full_title = f"Lecture {number}. {title}"
            print(f"\n--- [{number}/{total}] {full_title} ---")

            ap = lecture_artifact_paths(lectures_dir, number)
            if skip_completed and ap["deck"].exists():
                print(f"  -> already complete, loading from disk")
                outline = json.loads(ap["outline"].read_text())
                deck = json.loads(ap["deck"].read_text())
                evaluation = json.loads(ap["evaluation"].read_text())
                hist = json.loads(ap["improve_history"].read_text()) if ap["improve_history"].exists() else []
                slides = deck.get("slides", [])
                run_items.append({
                    "number": number,
                    "title": title,
                    "outline": outline,
                    "deck": deck,
                    "evaluation": evaluation,
                    "initialScore": evaluation.get("alignmentScore"),
                    "improveRoundsUsed": len(hist),
                    "improveHistory": hist,
                    "slideCount": len(slides),
                    "artifacts": {k: str(v) for k, v in ap.items()},
                })
                show_title(f"Lecture {number} — loaded from disk (score {evaluation.get('alignmentScore')})", level=4)
                continue

            description = build_lecture_description(lecture, module, course_context)
            outline = generate_outline(title=full_title, description=description, target_slides=slides_per_lecture)
            deck = generate_slides(outline, lecture_title=outline.get("title", full_title))
            slides = deck.get("slides", [])
            evaluation = evaluate_deck(
                lecture_title=outline.get("title", full_title),
                slides=slides,
            )
            initial_evaluation = evaluation

            improve_result = run_improve_loop(
                lecture_title=outline.get("title", full_title),
                outline=outline,
                deck=deck,
                evaluation=evaluation,
                auto_improve=auto_improve,
                reevaluate=reevaluate_after_improve,
            )
            deck = improve_result["deck"]
            evaluation = improve_result["evaluation"]
            slides = deck.get("slides", [])
            ap["outline"].write_text(json.dumps(outline, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            ap["deck"].write_text(json.dumps(deck, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            ap["evaluation"].write_text(json.dumps(evaluation, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            ap["improve_history"].write_text(json.dumps(improve_result["improve_history"], indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            save_slide_eval_preview_html(deck=deck, evaluation=evaluation, lecture_title=full_title, path=ap["preview"])

            run_items.append({
                "number": number,
                "title": title,
                "outline": outline,
                "deck": deck,
                "evaluation": evaluation,
                "initialScore": improve_result.get("initial_score"),
                "improveRoundsUsed": improve_result.get("rounds_used", 0),
                "improveHistory": improve_result.get("improve_history", []),
                "slideCount": len(slides),
                "artifacts": {k: str(v) for k, v in ap.items()},
            })

            show_title(
                f"Lecture {number} — {improve_result.get('initial_score')} → {evaluation.get('alignmentScore')}",
                level=4,
            )
            show_score_progress_charts(
                initial_evaluation=initial_evaluation,
                improve_result=improve_result,
                lecture_title=f"Lecture {number}: {title}",
                save_stem=ap["score_progress_figure"],
            )

    if smoke_test_first and len(plan_lectures) > 1 and max_lectures is None:
        process_lectures(plan_lectures[:1], "=== Step 4 Phase 1: smoke test (lecture 1) ===")
        print("\n✓ Smoke test passed — continuing with remaining lectures\n")
        process_lectures(plan_lectures[1:], f"=== Step 4 Phase 2: remaining {len(plan_lectures) - 1} lecture(s) ===")
    else:
        process_lectures(plan_lectures, f"=== Step 4: processing {total} lecture(s) ===")

    run_bundle = {
        "generatedAt": datetime.now(timezone.utc).isoformat(),
        "courseTitle": course_title,
        "lecturePlan": lecture_plan,
        "lectures": run_items,
    }
    paths["full_run_json"].write_text(json.dumps(run_bundle, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    save_full_run_markdown(run_bundle, paths["full_run_markdown"])
    return run_bundle


FULL_RUN = run_pipeline_for_all_lectures(
    lecture_plan=LECTURE_PLAN,
    module=selected_module,
    course_context=COURSE_CONTEXT,
    course_title=COURSE_TITLE,
    paths=LECTURE_PLAN_PATHS,
)

show_title("Full run complete", level=3)
show_fields({
    "lectures_processed": len(FULL_RUN["lectures"]),
    "full_run_json": str(LECTURE_PLAN_PATHS["full_run_json"]),
    "full_run_markdown": str(LECTURE_PLAN_PATHS["full_run_markdown"]),
    "lectures_dir": str(LECTURE_PLAN_PATHS["lectures_dir"]),
})
show_table(
    [
        [item["number"], item["title"], item.get("initialScore"),
         item["evaluation"].get("alignmentScore"), item.get("improveRoundsUsed", 0)]
        for item in FULL_RUN["lectures"]
    ],
    headers=["#", "Lecture", "Score init", "Score final", "Rounds"],
)
print("Saved full run artifacts. Run Step 5 to export PDF and PPTX.")

=== Step 4 Phase 1: smoke test (lecture 1) ===

--- [1/10] Lecture 1. Introduction to Software Engineering and Object-Oriented Fundamentals ---
  -> already complete, loading from disk



✓ Smoke test passed — continuing with remaining lectures

=== Step 4 Phase 2: remaining 9 lecture(s) ===

--- [2/10] Lecture 2. Advanced Object-Oriented Mechanisms: Inheritance and Polymorphism ---
  -> already complete, loading from disk



--- [3/10] Lecture 3. Genericity and Design by Contract ---
  -> already complete, loading from disk



--- [4/10] Lecture 4. Fundamental Design Patterns I: Creational and Structural Patterns ---
  -> already complete, loading from disk



--- [5/10] Lecture 5. Fundamental Design Patterns II: Behavioral Patterns ---
  -> already complete, loading from disk


...


--- [8/10] Lecture 8. Agile Development Methods and Practices ---
  -> already complete, loading from disk



--- [9/10] Lecture 9. Software Quality and Testing Strategies ---
  -> already complete, loading from disk



--- [10/10] Lecture 10. Software Engineering Process and Project Management ---
  -> already complete, loading from disk


#,Lecture,Score init,Score final,Rounds
1,Introduction to Software Engineering and Object-Oriented Fundamentals,100,100,0
2,Advanced Object-Oriented Mechanisms: Inheritance and Polymorphism,100,100,0
3,Genericity and Design by Contract,81,81,3
4,Fundamental Design Patterns I: Creational and Structural Patterns,100,100,1
5,Fundamental Design Patterns II: Behavioral Patterns,100,100,1
6,Software Architecture: Structuring Large-Scale Systems,100,100,1
7,Requirements Engineering Fundamentals,100,100,0
8,Agile Development Methods and Practices,100,100,0
9,Software Quality and Testing Strategies,100,100,1
10,Software Engineering Process and Project Management,100,100,2


Saved full run artifacts. Run Step 5 to export PDF and PPTX.


## Step 5 — Export slides to PDF and PPTX

Reads each `lecture-XX-deck.json` from disk and exports to PDF and PPTX.

Re-run this cell any time you change the exporters (Steps 1–3.5 do not need to re-run).
Requires Step 0.5 to have set `LECTURE_PLAN_PATHS`.

In [14]:
def export_slides_from_disk(lectures_dir: Path) -> None:
    deck_files = sorted(lectures_dir.glob("lecture-*-deck.json"))
    if not deck_files:
        print(f"No deck files found in {lectures_dir}")
        return

    print(f"=== Step 5: exporting {len(deck_files)} lecture(s) ===")
    rows = []
    for deck_path in deck_files:
        m = re.search(r"lecture-(\d+)-deck\.json", deck_path.name)
        number = int(m.group(1)) if m else 0

        deck = json.loads(deck_path.read_text(encoding="utf-8"))
        slides = deck.get("slides", [])

        outline_path = deck_path.with_name(deck_path.name.replace("-deck.json", "-outline.json"))
        if outline_path.exists():
            outline = json.loads(outline_path.read_text(encoding="utf-8"))
            lecture_title = outline.get("title") or f"Lecture {number:02d}"
        else:
            lecture_title = f"Lecture {number:02d}"

        pdf_path = deck_path.with_name(deck_path.name.replace("-deck.json", "-slides.pdf"))
        pptx_path = deck_path.with_name(deck_path.name.replace("-deck.json", "-slides.pptx"))

        pdf_path.write_bytes(export_deck_to_pdf(lecture_title, slides))
        pptx_path.write_bytes(export_deck_to_pptx(lecture_title, slides))
        rows.append([number, lecture_title, len(slides), pdf_path.name, pptx_path.name])
        print(f"  [{number:02d}] {lecture_title} — {len(slides)} slides")

    show_table(rows, headers=["#", "Lecture", "Slides", "PDF", "PPTX"])
    print(f"Done. Files written to {lectures_dir}")


export_slides_from_disk(LECTURE_PLAN_PATHS["lectures_dir"])

=== Step 5: exporting 10 lecture(s) ===


  [01] Lecture 1: Introduction to Software Engineering and Object-Oriented Fundamentals — 18 slides


  [02] Lecture 2. Advanced Object-Oriented Mechanisms: Inheritance and Polymorphism — 18 slides


  [03] Lecture 3: Genericity and Design by Contract — 18 slides


  [04] Lecture 4. Fundamental Design Patterns I: Creational and Structural Patterns — 18 slides


...

  [07] Lecture 7: Requirements Engineering Fundamentals — 20 slides
  [08] Lecture 8: Agile Development Methods and Practices — 19 slides


  [09] Lecture 9: Software Quality Assurance and Testing Strategies — 18 slides


  [10] Lecture 10: Software Engineering Best Practices and Professional Development — 18 slides


#,Lecture,Slides,PDF,PPTX
1,Lecture 1: Introduction to Software Engineering and Object-Oriented Fundamentals,18,lecture-01-slides.pdf,lecture-01-slides.pptx
2,Lecture 2. Advanced Object-Oriented Mechanisms: Inheritance and Polymorphism,18,lecture-02-slides.pdf,lecture-02-slides.pptx
3,Lecture 3: Genericity and Design by Contract,18,lecture-03-slides.pdf,lecture-03-slides.pptx
4,Lecture 4. Fundamental Design Patterns I: Creational and Structural Patterns,18,lecture-04-slides.pdf,lecture-04-slides.pptx
5,Lecture 5. Fundamental Design Patterns II: Behavioral Patterns,23,lecture-05-slides.pdf,lecture-05-slides.pptx
6,Lecture 6. Software Architecture: Structuring Large-Scale Systems,18,lecture-06-slides.pdf,lecture-06-slides.pptx
7,Lecture 7: Requirements Engineering Fundamentals,20,lecture-07-slides.pdf,lecture-07-slides.pptx
8,Lecture 8: Agile Development Methods and Practices,19,lecture-08-slides.pdf,lecture-08-slides.pptx
9,Lecture 9: Software Quality Assurance and Testing Strategies,18,lecture-09-slides.pdf,lecture-09-slides.pptx
10,Lecture 10: Software Engineering Best Practices and Professional Development,18,lecture-10-slides.pdf,lecture-10-slides.pptx


Done. Files written to /home/ilgiz/sci/lectura/output/course-generation/2025-mcsse-se-01/lectures


In [15]:
# --- Step 6: final scores per lecture and per criterion (reads saved artifacts, no regeneration) ---
import json as _json
from pathlib import Path as _Path

_OUTPUT_DIR = NOTEBOOK_DIR / "output" / "course-generation"
_slug = module_slug(selected_module["code"])
_full_run_path = _OUTPUT_DIR / f"{_slug}-full-run.json"

if not _full_run_path.exists():
    print(f"No full-run JSON at {_full_run_path}. Run Step 4 first.")
else:
    _run = _json.loads(_full_run_path.read_text(encoding="utf-8"))
    _lectures = _run.get("lectures", [])
    _criteria = ["correctness", "completeness", "clarity", "readyToTeach"]
    _criteria_labels = ["Correctness", "Completeness", "Clarity", "Ready-to-teach"]

    show_title("Final scores", level=3)

    # Overall table
    show_table(
        [
            [
                item["number"],
                item["title"],
                item.get("slideCount", "—"),
                item.get("initialScore", "—"),
                (item.get("evaluation") or {}).get("alignmentScore", "—"),
                item.get("improveRoundsUsed", 0),
            ]
            for item in _lectures
        ],
        headers=["#", "Lecture", "Slides", "Score init", "Score final", "Rounds"],
    )

    # Per-criterion table
    show_title("Scores by criterion (1–5)", level=4)
    show_table(
        [
            [
                item["number"],
                item["title"],
            ] + [
                (item.get("evaluation") or {}).get("scores", {}).get(k, "—")
                for k in _criteria
            ] + [
                (item.get("evaluation") or {}).get("alignmentScore", "—"),
            ]
            for item in _lectures
        ],
        headers=["#", "Lecture"] + _criteria_labels + ["Overall /100"],
    )

    _avg = sum((item.get("evaluation") or {}).get("alignmentScore", 0) for item in _lectures) / max(len(_lectures), 1)
    _crit_avgs = {
        k: sum((item.get("evaluation") or {}).get("scores", {}).get(k, 0) for item in _lectures) / max(len(_lectures), 1)
        for k in _criteria
    }
    print(f"Average overall: {_avg:.1f}/100")
    for k, lbl in zip(_criteria, _criteria_labels):
        print(f"  {lbl}: {_crit_avgs[k]:.2f}/5")


#,Lecture,Slides,Score init,Score final,Rounds
1,Introduction to Software Engineering and Object-Oriented Fundamentals,18,100,100,0
2,Advanced Object-Oriented Mechanisms: Inheritance and Polymorphism,18,100,100,0
3,Genericity and Design by Contract,18,81,81,3
4,Fundamental Design Patterns I: Creational and Structural Patterns,18,100,100,1
5,Fundamental Design Patterns II: Behavioral Patterns,23,100,100,1
6,Software Architecture: Structuring Large-Scale Systems,18,100,100,1
7,Requirements Engineering Fundamentals,20,100,100,0
8,Agile Development Methods and Practices,19,100,100,0
9,Software Quality and Testing Strategies,18,100,100,1
10,Software Engineering Process and Project Management,18,100,100,2


#,Lecture,Correctness,Completeness,Clarity,Ready-to-teach,Overall /100
1,Introduction to Software Engineering and Object-Oriented Fundamentals,5,5,5,5,100
2,Advanced Object-Oriented Mechanisms: Inheritance and Polymorphism,5,5,5,5,100
3,Genericity and Design by Contract,4,4,5,4,81
4,Fundamental Design Patterns I: Creational and Structural Patterns,5,5,5,5,100
5,Fundamental Design Patterns II: Behavioral Patterns,5,5,5,5,100
6,Software Architecture: Structuring Large-Scale Systems,5,5,5,5,100
7,Requirements Engineering Fundamentals,5,5,5,5,100
8,Agile Development Methods and Practices,5,5,5,5,100
9,Software Quality and Testing Strategies,5,5,5,5,100
10,Software Engineering Process and Project Management,5,5,5,5,100


Average overall: 98.1/100
  Correctness: 4.90/5
  Completeness: 4.90/5
  Clarity: 5.00/5
  Ready-to-teach: 4.90/5
